In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, precision_recall_curve
)
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import DBSCAN
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDOneClassSVM
from sklearn.pipeline import make_pipeline
import tensorflow as tf
from tensorflow import keras

FEATURE_COLS = [
    'login_count', 'device_event_count', 'http_event_count',
    'unique_pc_count', 'after_hours_count', 'is_weekend',
    'first_login_hour', 'last_activity_hour'
]


# combined_df contains: FEATURE_COLS + is_threat + if_pred + svm_pred + ae_pred +
#                        dbscan_pred + ensemble_2plus + ensemble_3plus




def calculate_metrics(combined_df):
    print("--- Challenge 6: Precision, Recall, F1 ---\n")

    y_true = combined_df['is_threat'].values
    models = {
        'Isolation Forest':      combined_df['if_pred'].values,
        'One-Class SVM':         combined_df['svm_pred'].values,
        'Autoencoder':           combined_df['ae_pred'].values,
        'DBSCAN':                combined_df['dbscan_pred'].values,
        'Ensemble (2+/4 agree)': combined_df['ensemble_2plus'].values,
    }

    results = []
    for name, y_pred in models.items():
        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)
        f = f1_score(y_true, y_pred, zero_division=0)
        results.append({'Model': name, 'Precision': round(p, 4),
                        'Recall': round(r, 4), 'F1-Score': round(f, 4)})
        print(f"{name}:")
        print(classification_report(y_true, y_pred,
              target_names=['Normal', 'Threat'], zero_division=0))

    metrics_df = pd.DataFrame(results)
    print("\nSummary:")
    print(metrics_df)
    return metrics_df




def plot_roc_curves(combined_df):
    print("\n--- Challenge 7: ROC-AUC Curves ---")

    y_true = combined_df['is_threat'].values

    X = combined_df[FEATURE_COLS].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    
    if_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    if_model.fit(X_scaled)
    if_scores = -if_model.score_samples(X_scaled)

    
    svm_pipeline = make_pipeline(
        Nystroem(kernel='rbf', gamma=None, n_components=500, random_state=42),
        SGDOneClassSVM(nu=0.05, random_state=42)
    )
    svm_pipeline.fit(X_scaled)
    svm_scores = -svm_pipeline.decision_function(X_scaled)

    
    tf.keras.utils.set_random_seed(42)
    input_dim = X_scaled.shape[1]
    input_layer = keras.Input(shape=(input_dim,))
    enc = keras.layers.Dense(8, activation='relu')(input_layer)
    enc = keras.layers.Dense(4, activation='relu')(enc)
    dec = keras.layers.Dense(8, activation='relu')(enc)
    dec = keras.layers.Dense(input_dim, activation='linear')(dec)
    ae = keras.Model(input_layer, dec)
    ae.compile(optimizer='adam', loss='mse')
    ae.fit(X_scaled, X_scaled, epochs=20, batch_size=256,
           validation_split=0.1, verbose=0)
    reconstructed = ae.predict(X_scaled, verbose=0)
    ae_scores = np.mean(np.power(X_scaled - reconstructed, 2), axis=1)

    # DBSCAN doesn't produce a continuous anomaly score natively —
    # use distance to nearest sampled cluster center as a proxy score
    print("Computing DBSCAN-based distance scores...")
    pca = PCA(n_components=5, random_state=42)
    X_reduced = pca.fit_transform(X_scaled)

    combined_reset = combined_df.reset_index(drop=True)
    np.random.seed(42)
    sample_idx = (
        combined_reset.groupby('user', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 15), random_state=42), include_groups=False)
        .index
    )
    remaining = list(set(range(len(combined_reset))) - set(sample_idx))
    extra_needed = max(0, 50000 - len(sample_idx))
    if extra_needed > 0:
        extra_idx = np.random.choice(remaining, size=extra_needed, replace=False)
        sample_idx = list(sample_idx) + list(extra_idx)

    X_sample = X_reduced[sample_idx]
    dbscan_model = DBSCAN(eps=0.8, min_samples=5, algorithm='ball_tree', n_jobs=-1)
    sample_labels = dbscan_model.fit_predict(X_sample)

    # Distance to nearest core point as anomaly proxy score (higher = more anomalous)
    knn_dist = KNeighborsClassifier(n_neighbors=3, algorithm='ball_tree', n_jobs=-1)
    knn_dist.fit(X_sample, sample_labels)
    neighbor_dists, _ = knn_dist.kneighbors(X_reduced)
    dbscan_scores = neighbor_dists.mean(axis=1)  

    # Ensemble score = average of all 4, normalized
    ensemble_scores = (
        if_scores / if_scores.max() +
        svm_scores / svm_scores.max() +
        ae_scores / ae_scores.max() +
        dbscan_scores / dbscan_scores.max()
    ) / 4

    plt.figure(figsize=(10, 6))
    for name, scores in [('Isolation Forest', if_scores),
                          ('One-Class SVM', svm_scores),
                          ('Autoencoder', ae_scores),
                          ('DBSCAN', dbscan_scores),
                          ('Ensemble', ensemble_scores)]:
        fpr, tpr, _ = roc_curve(y_true, scores)
        auc = roc_auc_score(y_true, scores)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves — Anomaly Detection Models')
    plt.legend()
    plt.tight_layout()
    plt.savefig('roc_curves.png')
    plt.show()
    print("Saved: roc_curves.png")

    return if_scores, svm_scores, ae_scores, dbscan_scores, ensemble_scores



def analyze_false_positives(combined_df):
    print("\n--- Challenge 8: False Positive Analysis ---")

    fp = combined_df[(combined_df['ensemble_2plus'] == 1) &
                     (combined_df['is_threat'] == 0)]

    print(f"Total false positives: {len(fp)}")
    print(f"False positive rate: {len(fp)/len(combined_df[combined_df['is_threat']==0])*100:.2f}%")

    print("\nFalse positive feature averages:")
    print(fp[FEATURE_COLS].mean().round(2))

    print("\nNormal (true negative) feature averages:")
    tn = combined_df[(combined_df['ensemble_2plus'] == 0) & (combined_df['is_threat'] == 0)]
    print(tn[FEATURE_COLS].mean().round(2))

    fp_users = fp.groupby('user').size().reset_index(name='fp_count')
    fp_users = fp_users.sort_values('fp_count', ascending=False)
    print("\nTop 10 users generating most false positives:")
    print(fp_users.head(10))

    print("\nMost common after_hours_count in false positives:")
    print(fp['after_hours_count'].describe())

    return fp




def compare_models(combined_df, if_scores, svm_scores, ae_scores, dbscan_scores, ensemble_scores):
    print("\n--- Challenge 9: Model Comparison Table ---")

    y_true = combined_df['is_threat'].values

    comparison = []
    for name, y_pred, scores in [
        ('Isolation Forest',      combined_df['if_pred'].values, if_scores),
        ('One-Class SVM',         combined_df['svm_pred'].values, svm_scores),
        ('Autoencoder',           combined_df['ae_pred'].values, ae_scores),
        ('DBSCAN',                combined_df['dbscan_pred'].values, dbscan_scores),
        ('Ensemble (2+/4 agree)', combined_df['ensemble_2plus'].values, ensemble_scores),
    ]:
        comparison.append({
            'Model':     name,
            'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
            'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
            'F1-Score':  round(f1_score(y_true, y_pred, zero_division=0), 4),
            'ROC-AUC':   round(roc_auc_score(y_true, scores), 4),
        })

    comparison_df = pd.DataFrame(comparison)
    print(comparison_df.to_string(index=False))
    return comparison_df




def optimize_threshold(combined_df, ensemble_scores):
    print("\n--- Challenge 10: Risk Score Threshold Optimization ---")

    y_true = combined_df['is_threat'].values

    precisions, recalls, thresholds = precision_recall_curve(y_true, ensemble_scores)

    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]

    print(f"Optimal threshold: {best_threshold:.4f}")
    print(f"At optimal threshold:")
    print(f"  Precision: {precisions[best_idx]:.4f}")
    print(f"  Recall:    {recalls[best_idx]:.4f}")
    print(f"  F1-Score:  {f1_scores[best_idx]:.4f}")

    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.plot(thresholds, precisions[:-1], label='Precision')
    plt.plot(thresholds, recalls[:-1], label='Recall')
    plt.plot(thresholds, f1_scores[:-1], label='F1-Score')
    plt.axvline(best_threshold, color='red', linestyle='--', label='Optimal threshold')
    plt.xlabel('Threshold')
    plt.title('Precision / Recall / F1 vs Threshold')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(recalls, precisions)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')

    plt.tight_layout()
    plt.savefig('threshold_optimization.png')
    plt.show()
    print("Saved: threshold_optimization.png")

    return best_threshold





# Needs combined_df from the evaluate_detection_full() run (obvious or subtle)
metrics_df = calculate_metrics(combined_df)
if_scores, svm_scores, ae_scores, dbscan_scores, ens_scores = plot_roc_curves(combined_df)
fp_df = analyze_false_positives(combined_df)
comparison_df = compare_models(combined_df, if_scores, svm_scores, ae_scores, dbscan_scores, ens_scores)
best_threshold = optimize_threshold(combined_df, ens_scores)